## Installing necessary libraries

Cohere provides free trial keys to use their LLMs. So generate one trial key from dashboard.cohere.com

In [ ]:
!pip install langchain-cohere langchain pdfminer.six chromadb langchain-community langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 873.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9

langchain-cohere: Enables integration of Cohere's language models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

In [ ]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, AIMessage,HumanMessage
from google.colab import userdata
os.environ["COHERE_API_KEY"] = userdata.get('COHERE_KEY')
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from pydantic import BaseModel, Field
from langchain_cohere import ChatCohere
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_community.vectorstores import Chroma
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel,RunnablePassthrough
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

## VectorDB setup

In [ ]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db"

# Initialize Cohere embeddings with the specified model
# "embed-english-v3.0" is a pre-trained English language embedding model by Cohere
# The user_agent parameter specifies the tool or library using the Cohere API, in this case, LangChain
embedding = CohereEmbeddings(
    model="embed-english-v3.0",
    user_agent="langchain"
)

We are processing 2 research papers on transformers and yolo. You can use the PDFs

In [ ]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

In [ ]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [ ]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate a

## Chain Setup

In [ ]:
# Initialize an LLM instance using Cohere's "command-r" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatCohere(model="command-a-plus-05-2026", temperature=0)

In [ ]:
# Define a prompt template for generating answers based on a given context and question
prompt_str = """Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

# Use a prompt that includes a MessagesPlaceholder variable under the name "chat_history".
# This allows us to pass in a list of Messages to the prompt using the "chat_history" input key,
# and these messages will be inserted after the system message and before the human message containing the
# latest question.

prompt_history_aware = ChatPromptTemplate.from_messages([
    ("system", prompt_str),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# create_history_aware_retriever constructs a chain that accepts keys input and chat_history as input, and has the same output schema as a retriever
history_aware_retriever = create_history_aware_retriever(
    llm, vectordb.as_retriever(), prompt_history_aware
)

Now its time to build our final rag_chain with create_retrieval_chain. This chain applies the history_aware_retriever and question_answer_chain created with create_stuff_documents_chain in sequence, retaining intermediate outputs such as the retrieved context for convenience. It has input keys input and chat_history, and includes input, chat_history, context, and answer in its output.



In [ ]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. If you don't know the answer,
say that you don't know. Use three sentences maximum and keep theanswer concise.

{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# create_retrieval_chain combines history aware retriever and the qa chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
# Load a empty chat_history list
chat_history = []

# Invoking the chain
question = "What is YOLO?"
response = rag_chain.invoke({"input": question, "chat_history": chat_history})

# Appeding question and response answers
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response["answer"]),
    ]
)

In [ ]:
#check chat_history
chat_history

[HumanMessage(content='What is YOLO?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='YOLO (You Only Look Once) is a unified, end‑to‑end convolutional neural network for object detection that predicts multiple bounding boxes and class probabilities in a single forward pass. It runs at real‑time speeds (over 150\u202ffps) and is designed to be simple, fast, and highly generalizable across domains. The model divides an image into a grid, each cell predicting boxes and confidence scores, allowing it to reason globally about the entire image.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
second_question = "What are the tasks YOLO can be used in?"
response = rag_chain.invoke({"input": second_question, "chat_history": chat_history})

print(response["answer"])

YOLO can be used for real-time object detection in streaming video and webcam feeds, enabling interactive tracking of objects as they move. It is suitable for detection tasks in diverse domains, including natural photographs and artwork, where fast and robust detection is required. Its speed and generalization make it ideal for applications such as surveillance, robotics, and augmented reality.


To automate the inserting and updating of chat history. And have a session id that can be unique for a user

In [ ]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
conversational_rag_chain.invoke(
    {"input": "What is Transformers?"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)["answer"]

'Transformers is a neural architecture that replaces recurrence with self‑attention, enabling global dependencies between input and output positions while allowing full parallelization. It consists of an encoder and decoder stack of multi‑head self‑attention and point‑wise feed‑forward layers, each with residual connections and layer normalization. This design achieves state‑of‑the‑art performance on tasks like machine translation with significantly reduced training time.'

In [ ]:
# This will output input, chat_history, contexts and answer
conversational_rag_chain.invoke(
    {"input": "Transformers vs YOLO"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)

{'input': 'Transformers vs YOLO',
 'chat_history': [HumanMessage(content='What is Transformers?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Transformers is a neural architecture that replaces recurrence with self‑attention, enabling global dependencies between input and output positions while allowing full parallelization. It consists of an encoder and decoder stack of multi‑head self‑attention and point‑wise feed‑forward layers, each with residual connections and layer normalization. This design achieves state‑of‑the‑art performance on tasks like machine translation with significantly reduced training time.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='lem, straight from image pixels to bounding box coordi- nates and class probabilities. Using our system, you only look once (YOLO) at an image to predict what objects are present and where

In [ ]:
# Check the chat_history of the store dict
for message in store["User_1"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

User: What is Transformers?

AI: Transformers is a neural architecture that replaces recurrence with self‑attention, enabling global dependencies between input and output positions while allowing full parallelization. It consists of an encoder and decoder stack of multi‑head self‑attention and point‑wise feed‑forward layers, each with residual connections and layer normalization. This design achieves state‑of‑the‑art performance on tasks like machine translation with significantly reduced training time.

User: Transformers vs YOLO

AI: The provided context does not contain information about Transformers, so I cannot compare them to YOLO.

